In [175]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
# Загружаем сразу в DataFrame
df = sns.load_dataset('titanic')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
None


**survived** - Выживание (0 = погиб, 1 = выжил) - целевая переменная

**pclass** - Класс билета (1 = первый класс, 2 = второй класс, 3 = третий класс)

**sex** - Пол (male = мужчина, female = женщина)

**age** - Возраст в годах

**sibsp** - Количество братьев/сестер/супругов на борту

**parch** - Количество родителей/детей на борту

**fare** - Стоимость билета в фунтах стерлингов

**embarked** - Порт посадки (C = Шербур, Q = Квинстаун, S = Саутгемптон)

**class** - Класс билета (категориальный: First, Second, Third)

**who** - Демографическая категория (man = мужчина, woman = женщина, child = ребенок)

**adult_male** - Взрослый мужчина? (True/False)

**deck** - Палуба каюты (A,B,C,D,E,F,G - где A ближе к шлюпкам)

**embark_town** - Город посадки (Cherbourg, Queenstown, Southampton)

**alive** - Выжил (yes/no - текстовый аналог survived)

**alone** - Путешествовал один? (True если sibsp + parch = 0)

In [176]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [177]:
mode1=df['embarked'].mode()[0]
mode2=df['embark_town'].mode()[0]
df['embarked'] = df['embarked'].fillna(mode1)
df['embark_town'] = df['embark_town'].fillna(mode2)
df.drop(['deck', 'alive'], axis=1, inplace=True)

In [178]:
df = df[df['fare'] < 200]

In [179]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            175
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
embark_town      0
alone            0
dtype: int64

Давай попробуем с помощью регрессии заполнить пропущенный возраст

In [180]:
df_nan = df[df['age'].isnull()]
df_full = df[~df['age'].isnull()]

In [181]:
print(f'Размер данных: {df_nan.shape}')
print(f'Размер данных: {df_full.shape}')

Размер данных: (175, 13)
Размер данных: (696, 13)


In [182]:
X_train, X_test, y_train = df_full.drop('age', axis=1), df_nan.drop('age', axis=1), df_full['age']

In [183]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,Southampton,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,Southampton,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,True


In [184]:
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
var1_prep = make_column_transformer(
    (StandardScaler(), ['fare']),
    (OneHotEncoder(), ['sex', 'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alone']))
print(var1_prep)

ColumnTransformer(transformers=[('standardscaler', StandardScaler(), ['fare']),
                                ('onehotencoder', OneHotEncoder(),
                                 ['sex', 'embarked', 'class', 'who',
                                  'adult_male', 'embark_town', 'alone'])])


In [185]:
from sklearn.linear_model import LinearRegression
Pipeline_1_regression = Pipeline([('scaler', var1_prep), ('lin_reg', LinearRegression())])
Pipeline_1_regression.fit(X_train, y_train)

,steps,"[('scaler', ...), ('lin_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('standardscaler', ...), ('onehotencoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [186]:
y_test = Pipeline_1_regression.predict(X_test)
y_test = np.floor(y_test)

In [187]:
null_indices = df[df['age'].isnull()].index
df.loc[null_indices, 'age'] = y_test

In [188]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,Southampton,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,True
888,0,3,female,26.0,1,2,23.4500,S,Third,woman,False,Southampton,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,True


In [189]:
from sklearn.model_selection import train_test_split
from sklearn import model_selection
X, y = df.drop('survived', axis=1), df['survived']
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, stratify=y, random_state=42)

Начнём классификацию. 

In [190]:
# Логистическая регрессия
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn import metrics
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Pipeline_logreg_1 = Pipeline([('scaler', var1_prep), ('log_reg', LogisticRegression(random_state=42))])
cv_metrics = model_selection.cross_validate(
    estimator=Pipeline_logreg_1,
    scoring='f1',
    cv=kf,
    X=X_train,
    y=y_train
)
display(cv_metrics)
Pipeline_logreg_1.fit(X_train, y_train)
y_predict_1 = Pipeline_logreg_1.predict(X_test)
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics['test_score']), 3)}')
print(f'Значение F1-score на тестовой выборке {np.round(metrics.f1_score(y_test, y_predict_1), 3)}')

{'fit_time': array([0.09599972, 0.06300116, 0.08700061, 0.06099963, 0.08100009]),
 'score_time': array([0.01800036, 0.03899908, 0.01799893, 0.03000045, 0.03300071]),
 'test_score': array([0.68817204, 0.73584906, 0.71910112, 0.73786408, 0.74      ])}

Среднее значение F1-score на валидационных фолдах: 0.724
Значение F1-score на тестовой выборке 0.623


In [191]:
from sklearn.model_selection import GridSearchCV
param_grid = [{'log_reg__penalty' : ['l1'],
               'log_reg__solver' : ['liblinear', 'saga'],
               'log_reg__C' : [0.001, 0.01, 0.1, 1]},
              {'log_reg__penalty': ['l2'],
               'log_reg__solver' : ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
               'log_reg__C' : [0.001, 0.01, 0.1, 1]}]

grid_search_1 = GridSearchCV(
    estimator=Pipeline_logreg_1,
    param_grid=param_grid,
    cv=kf,
    n_jobs=-1,
    scoring='f1',
    return_train_score=True
    
)
%time grid_search_1.fit(X_train, y_train)

CPU times: total: 844 ms
Wall time: 10.1 s


,estimator,Pipeline(step...m_state=42))])
,param_grid,"[{'log_reg__C': [0.001, 0.01, ...], 'log_reg__penalty': ['l1'], 'log_reg__solver': ['liblinear', 'saga']}, {'log_reg__C': [0.001, 0.01, ...], 'log_reg__penalty': ['l2'], 'log_reg__solver': ['newton-cg', 'lbfgs', ...]}]"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('standardscaler', ...), ('onehotencoder', ...)]"


In [200]:
params_log = grid_search_1.best_params_
print(grid_search_1.best_score_)

0.7476040992548574


In [202]:
params_log

{'log_reg__C': 0.1, 'log_reg__penalty': 'l2', 'log_reg__solver': 'newton-cg'}

In [204]:
Pipeline_1_regression = Pipeline([('scaler', var1_prep), ('lin_reg', LogisticRegression(C=0.1, penalty='l2', solver='newton-cg'))])
Pipeline_1_regression.fit(X_train, y_train)
y_test_logreg = Pipeline_1_regression.predict(X_test)
print(metrics.classification_report(y_test, y_test_logreg))

              precision    recall  f1-score   support

           0       0.77      0.83      0.80       136
           1       0.68      0.59      0.63        82

    accuracy                           0.74       218
   macro avg       0.72      0.71      0.71       218
weighted avg       0.73      0.74      0.73       218



In [ ]:
# Давай деревья деревья



In [205]:
y_true = [1.23, 2.35, 2.75]

In [206]:
y_pred = [1.01, 12.3, 2.74]
from sklearn.metrics import mean_squared_error
print(np.sqrt(metrics.mean_squared_error(y_true, y_pred)))

5.746042116100439
